# Step 11 — Model Explainability and Error Analysis

Model explainability and error analysis were performed following
patient-grouped cross-validation.

Out-of-fold predictions generated during grouped cross-validation were
used for error analysis, ensuring that each evaluated prediction was
generated by a model that had not been trained on that patient's
validation fold.

Model interpretation was performed using:

- standardized Logistic Regression coefficients;
- Random Forest feature importance; and
- SHAP values for XGBoost.

Models used for global interpretation were fitted to the complete
proof-of-concept dataset. Consequently, these importance estimates are
descriptive and are not independent validation results.

Feature importance and SHAP values represent predictive associations,
not causal effects.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

DATA_PATH = "../results/model_features_2h.csv"
PRED_PATH = "../results/grouped_cv_predictions.csv"

df = pd.read_csv(DATA_PATH)
# Locate the predictions file from common notebook/project locations
pred_candidates = [
    Path(PRED_PATH),
    Path.cwd() / Path(PRED_PATH),
    Path.cwd() / "results" / Path(PRED_PATH).name,
    Path.cwd().parent / "results" / Path(PRED_PATH).name,
]

pred_path = next((path for path in pred_candidates if path.exists()), None)

if pred_path is None:
    raise FileNotFoundError(
        "Prediction file not found. Checked:\n"
        + "\n".join(str(path.resolve()) for path in pred_candidates)
    )

PRED_PATH = str(pred_path)
pred = pd.read_csv(pred_path)

print(f"Loaded predictions from: {pred_path.resolve()}")
print(f"Prediction rows: {len(pred)}")

ID_COLUMNS = ["subject_id", "hadm_id", "stay_id"]
TARGET = "future_deterioration"

features = [
    c for c in df.columns
    if c not in ID_COLUMNS + [TARGET]
]

X = df[features].copy()
y = df[TARGET].copy()

# Remove completely empty columns, if any
empty_cols = X.columns[X.isna().all()].tolist()

if empty_cols:
    X = X.drop(columns=empty_cols)

features = X.columns.tolist()

print("ICU stays:", len(df))
print("Patients:", df["subject_id"].nunique())
print("Predictors:", len(features))
print("Positive outcomes:", int(y.sum()))

def error_type(row):
    if row["actual"] == 1 and row["predicted_class"] == 1:
        return "True Positive"
    elif row["actual"] == 0 and row["predicted_class"] == 0:
        return "True Negative"
    elif row["actual"] == 0 and row["predicted_class"] == 1:
        return "False Positive"
    else:
        return "False Negative"


pred["error_type"] = pred.apply(
    error_type,
    axis=1
)

error_summary = (
    pred
    .groupby(["model", "error_type"])
    .size()
    .unstack(fill_value=0)
)

display(error_summary)

rows = []

for model in pred["model"].unique():

    temp = pred[
        pred["model"] == model
    ]

    tp = (
        temp["error_type"]
        == "True Positive"
    ).sum()

    tn = (
        temp["error_type"]
        == "True Negative"
    ).sum()

    fp = (
        temp["error_type"]
        == "False Positive"
    ).sum()

    fn = (
        temp["error_type"]
        == "False Negative"
    ).sum()

    rows.append({
        "model": model,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "false_alerts_per_true_alert":
            fp / tp if tp > 0 else np.nan,
        "missed_event_percent":
            100 * fn / (tp + fn)
            if (tp + fn) > 0
            else np.nan
    })

error_metrics = pd.DataFrame(rows)

display(
    error_metrics.round(3)
)

logistic = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )
])

logistic.fit(X, y)

coef = pd.DataFrame({
    "feature": features,
    "coefficient":
        logistic.named_steps["model"].coef_[0]
})

coef["absolute_coefficient"] = (
    coef["coefficient"].abs()
)

coef = coef.sort_values(
    "absolute_coefficient",
    ascending=False
)

display(
    coef[
        ["feature", "coefficient"]
    ].head(10)
)


rf = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=100,
            max_depth=4,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )
    )
])

rf.fit(X, y)

rf_importance = pd.DataFrame({
    "feature": features,
    "importance":
        rf.named_steps["model"].feature_importances_
})

rf_importance = (
    rf_importance
    .sort_values(
        "importance",
        ascending=False
    )
)

display(
    rf_importance[
        ["feature", "importance"]
    ].head(10)
)

spo2_features = [
    c for c in df.columns
    if "spo2" in c.lower()
]

print("SpO2 features:")
print(spo2_features)

print("\nSpO2 descriptive statistics:")

display(
    df[spo2_features].describe().T[
        [
            "count",
            "mean",
            "std",
            "min",
            "max"
        ]
    ]
)

spo2_features = [
    c for c in df.columns
    if "spo2" in c.lower()
]

print("SpO2 features found:")
print(spo2_features)

if spo2_features:
    display(
        df[spo2_features]
        .describe()
        .T
        [["count", "mean", "std", "min", "max"]]
        .round(2)
    )

Loaded predictions from: C:\Users\HARIKRISHNAN\Documents\Phoenix Code\default project\mimic-deterioration-ai\results\grouped_cv_predictions.csv
Prediction rows: 297
ICU stays: 99
Patients: 79
Predictors: 42
Positive outcomes: 18


error_type,False Negative,False Positive,True Negative,True Positive
model,,,,
Logistic Regression,2,16,65,16
Random Forest,2,5,76,16
XGBoost,2,5,76,16


,model,TP,TN,FP,FN,false_alerts_per_true_alert,missed_event_percent
0,Logistic Regression,16,65,16,2,1.000,11.111
1,Random Forest,16,76,5,2,0.312,11.111
2,XGBoost,16,76,5,2,0.312,11.111


,feature,coefficient
18,spo2_max,1.659923
25,spo2_std,-1.633542
11,spo2_min,-1.493736
28,diastolic_bp_last,-1.228036
39,spo2_count,-1.081915
41,temperature_count,-1.037743
12,systolic_bp_min,-0.984035
17,respiratory_rate_max,0.930624
33,systolic_bp_last,0.860067
31,respiratory_rate_last,-0.727150


,feature,importance
8,heart_rate_min,0.088370
1,heart_rate_mean,0.075358
3,respiratory_rate_mean,0.061719
15,heart_rate_max,0.060182
29,heart_rate_last,0.054070
22,heart_rate_std,0.043704
2,mean_bp_mean,0.042232
38,respiratory_rate_count,0.041435
5,systolic_bp_mean,0.041287
31,respiratory_rate_last,0.039779


SpO2 features:
['spo2_mean', 'spo2_min', 'spo2_max', 'spo2_std', 'spo2_last', 'spo2_count']

SpO2 descriptive statistics:


,count,mean,std,min,max
spo2_mean,78.0,96.738122,2.648764,85.0,100.0
spo2_min,78.0,94.692308,5.025410,64.0,100.0
spo2_max,78.0,98.205128,2.326354,87.0,100.0
spo2_std,76.0,1.762459,2.312182,0.0,18.0
spo2_last,78.0,96.538462,3.030976,83.0,100.0
spo2_count,78.0,3.987179,2.473200,1.0,17.0


SpO2 features found:
['spo2_mean', 'spo2_min', 'spo2_max', 'spo2_std', 'spo2_last', 'spo2_count']


,count,mean,std,min,max
spo2_mean,78.0,96.74,2.65,85.0,100.0
spo2_min,78.0,94.69,5.03,64.0,100.0
spo2_max,78.0,98.21,2.33,87.0,100.0
spo2_std,76.0,1.76,2.31,0.0,18.0
spo2_last,78.0,96.54,3.03,83.0,100.0
spo2_count,78.0,3.99,2.47,1.0,17.0
